# 🎨 AI Media OS

ComfyUI + image generation + face swap + face lock/consistency + background change + upscaling. Optional video generation.

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run All**.


In [ ]:
# @title 📥 STEP 1: Clone Repository from GitHub
import os, sys
from pathlib import Path

print("="*60)
print("🔧 Cloning AI Media OS...")
print("="*60)

# ⚠️ Replace this with YOUR OWN GitHub repo URL after you push this
# project's files there.
REPO_URL = "https://github.com/abhayraj01op/AI_Media_OS.git"

repo_path = Path('/content/AI_Media_OS')

if not repo_path.exists():
    !git clone {REPO_URL} /content/AI_Media_OS
    if not repo_path.exists():
        raise RuntimeError(
            "❌ Repository clone failed! Check that REPO_URL above is correct, "
            "that the repo is public, and that this Colab session has internet access."
        )
    print("✅ Repository cloned!")
else:
    %cd /content/AI_Media_OS
    !git fetch origin && git reset --hard origin/main
    if _exit_code != 0:
        raise RuntimeError(
            "❌ Failed to sync with GitHub (git fetch/reset failed) — see the error above. "
            "Quick fix: Runtime > Disconnect and delete runtime, then Run All again."
        )
    print("✅ Repository updated to the latest GitHub version!")

os.chdir('/content/AI_Media_OS')
!pip install -q -r requirements.txt
print("✅ Dependencies installed!")

print("\n⚠️  IMPORTANT: If you just pushed new code to GitHub, do")
print("   Runtime → Restart session BEFORE running Step 2/3 again.")


In [ ]:
# @title 🎛️ STEP 2 (optional): Choose which extra models to install
import os

# Z-Image Turbo — modern, fast, high quality (~19GB). ON by default.
os.environ["AI_MEDIA_ENABLE_ZIMAGE"] = "1"

# FLUX.1-schnell — alternative to Z-Image, also very good (~17GB). OFF by
# default since it's largely redundant with Z-Image — turn on if you want both.
os.environ["AI_MEDIA_ENABLE_FLUX"] = "0"

# Installs the LTX-Video custom NODE (not a model — see README for why).
# Leave this on unless you specifically don't want video generation at all.
os.environ["AI_MEDIA_ENABLE_VIDEO_NODE"] = "1"

print("Model selection set. Edit the values above and re-run this cell before Step 3 if you want to change anything.")


In [ ]:
# @title 🚀 STEP 3: Install (ComfyUI + face tools + upscaler + models)
import sys
sys.path.insert(0, '/content/AI_Media_OS')

from core.installer import MediaInstaller

print("\n🎯 Starting install — ComfyUI, SDXL, ReActor, InstantID, PuLID, IPAdapter, Real-ESRGAN")
print("⏳ Realistically 30-60+ minutes on first run (several GB of models). See README for details.\n")

installer = MediaInstaller()
installer.run()


In [ ]:
# @title ✅ STEP 4: Verify + Get Your Link
import json
from pathlib import Path

complete_path = Path('/content/drive/MyDrive/AI_Media_OS/logs/complete.json')
if not complete_path.exists():
    print("❗ No logs/complete.json found yet — Step 3 may not have finished running.")
else:
    with open(complete_path) as f:
        state = json.load(f)

    print("="*60)
    print("🔗 YOUR LINK")
    print("="*60)
    if state.get("comfyui_url"):
        print(f"ComfyUI: {state['comfyui_url']}")
    else:
        print("ComfyUI: not reachable — check logs/comfyui.log")

    print("\n" + "="*60)
    print("📦 COMPONENT STATUS")
    print("="*60)
    for cid, st in state.get("component_status", {}).items():
        icon = {"ready": "🟢", "partial": "🟡", "failed": "🔴", "skipped": "⏭️"}.get(st.get("status"), "⚪")
        print(f"  {cid:16} {icon} {st.get('status','?').upper():10} {st.get('detail','')[:80]}")

print("\n📁 Sab kuch Google Drive mein save hai (ComfyUI, models, custom nodes).")
print("💡 Agli baar bas Colab kholo aur Run All karo!")


In [ ]:
# @title 🩺 STEP 5: Debug ComfyUI (run only if the link is missing / blank)
from pathlib import Path

log_path = Path('/content/drive/MyDrive/AI_Media_OS/logs/comfyui.log')

if not log_path.exists():
    print("❌ No log file found yet — Step 3 may not have run, or ComfyUI never started at all.")
else:
    print(f"📄 Showing last 60 lines of: {log_path}\n")
    print("="*70)
    with open(log_path) as f:
        lines = f.readlines()
    print("".join(lines[-60:]))
    print("="*70)
    print("\n👉 Look for a Python traceback above. Copy that error and share it for an exact fix.")

import urllib.request
try:
    urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=5)
    print("\n✅ ComfyUI IS currently responding on port 8188 right now.")
except Exception as e:
    print(f"\n❌ ComfyUI is NOT responding on port 8188 right now: {e}")

import subprocess
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if r.returncode == 0:
    print("\n✅ GPU detected (nvidia-smi works):")
    print(r.stdout[:500])
else:
    print("\n❌ NO GPU on this runtime (nvidia-smi failed). This is almost always why ComfyUI")
    print("   won't start. Fix: Runtime menu > Change runtime type > T4 GPU > Save, then Run All again.")
